# Chapter 3: Option basics in code

Intrinsic value, time value, moneyness, and exercise cash flows for the three settlement types.
Notes: [[chapters/03-contract-specifications-and-option-terminology]]

In [1]:
def call_intrinsic(S, X):
    return max(0, S - X)

def put_intrinsic(S, X):
    return max(0, X - S)

def time_value(premium, intrinsic):
    return premium - intrinsic

def moneyness(kind, S, X):
    """Strict definition: ITM / ATM / OTM from the exercise price vs underlying price."""
    if S == X:
        return "at the money"
    itm = (X < S) if kind == "call" else (X > S)
    return "in the money" if itm else "out of the money"

## Intrinsic value and time value

In [2]:
iv = call_intrinsic(435, 400)
print(f"April 400 call at 50, underlying 435: intrinsic {iv}, time value {time_value(50, iv)}")
iv = put_intrinsic(62, 70)
print(f"October 70 put at 11, underlying 62: intrinsic {iv}, time value {time_value(11, iv)}")
print("70 calls at underlying 83:", call_intrinsic(83, 70), " 90 puts at underlying 83:", put_intrinsic(83, 90))

print()
print("Underlying   95 call   110 put")
for S in range(80, 130, 5):
    print(f"{S:>10} {call_intrinsic(S, 95):>9} {put_intrinsic(S, 110):>9}")

April 400 call at 50, underlying 435: intrinsic 35, time value 15
October 70 put at 11, underlying 62: intrinsic 8, time value 3
70 calls at underlying 83: 13  90 puts at underlying 83: 7

Underlying   95 call   110 put
        80         0        30
        85         0        25
        90         0        20
        95         0        15
       100         5        10
       105        10         5
       110        15         0
       115        20         0
       120        25         0
       125        30         0


## In, at and out of the money

In [3]:
S = 44
for kind, X in [("call", 40), ("put", 55), ("put", 40), ("call", 55)]:
    amount = call_intrinsic(S, X) if kind == "call" else put_intrinsic(S, X)
    print(f"stock {S}: {X} {kind} is {moneyness(kind, S, X)}  (intrinsic {amount})")

# The at-the-money convention: stock at 74, strikes every 5
S = 74
print()
print("stock 74, the 75 call:", moneyness("call", S, 75), "/ intrinsic", call_intrinsic(S, 75))
print("stock 74, the 75 put :", moneyness("put", S, 75), "/ intrinsic", put_intrinsic(S, 75))

stock 44: 40 call is in the money  (intrinsic 4)
stock 44: 55 put is in the money  (intrinsic 11)
stock 44: 40 put is out of the money  (intrinsic 0)
stock 44: 55 call is out of the money  (intrinsic 0)

stock 74, the 75 call: out of the money / intrinsic 0
stock 74, the 75 put : in the money / intrinsic 1


## Exercise cash flows by settlement type

In [4]:
# 1. Physical underlying: cash depends only on the exercise price. Positive = you receive, negative = you pay.
def physical(kind, role, contracts, X, shares_per=100):
    cash = X * shares_per * contracts
    exerciser_gets_cash = (kind == "put")
    gets_cash = exerciser_gets_cash if role == "exercise" else not exerciser_gets_cash
    return cash if gets_cash else -cash

print("Physical (stock):")
for kind, role, n, X in [("call","exercise",1,110), ("call","assigned",6,40), ("put","exercise",2,60), ("put","assigned",3,95)]:
    print(f"  {role} {n} x {X} {kind}: cash {physical(kind, role, n, X):>+9,}")

Physical (stock):
  exercise 1 x 110 call: cash   -11,000
  assigned 6 x 40 call: cash   +24,000
  exercise 2 x 60 put: cash   +12,000
  assigned 3 x 95 put: cash   -28,500


In [5]:
# 2. Futures position: the new futures position is at the exercise price; variation is paid vs today's futures price.
def futures_exercise(kind, role, contracts, X, futures_price, point_value=1000, margin=3000):
    long_future = (kind == "call") == (role == "exercise")   # exercise call or assigned put -> long
    signed = contracts if long_future else -contracts
    variation = (futures_price - X) * point_value * signed
    return ("long" if long_future else "short"), margin * contracts, variation

print("Futures (futures at 85, point value 1000, margin 3000):")
for kind, role, n, X in [("call","exercise",1,80), ("call","assigned",6,75), ("put","exercise",4,100), ("put","assigned",2,95)]:
    pos, m, v = futures_exercise(kind, role, n, X, 85)
    print(f"  {role} {n} x {X} {kind}: {pos} {n} at {X}, margin {m:>6,}, variation {v:>+9,}")

Futures (futures at 85, point value 1000, margin 3000):
  exercise 1 x 80 call: long 1 at 80, margin  3,000, variation    +5,000
  assigned 6 x 75 call: short 6 at 75, margin 18,000, variation   -60,000
  exercise 4 x 100 put: short 4 at 100, margin 12,000, variation   +60,000
  assigned 2 x 95 put: long 2 at 95, margin  6,000, variation   -20,000


In [6]:
# 3. Cash settlement (index): exerciser receives (S - X) for a call, (X - S) for a put, times point value and contracts.
#    The assigned seller pays the same amount.
def cash_settle(kind, role, contracts, X, index, point_value=500):
    value = (index - X) if kind == "call" else (X - index)
    amount = max(0, value) * point_value * contracts
    return amount if role == "exercise" else -amount

print("Cash (index 300, 500 per point):")
for kind, role, n, X in [("call","exercise",3,250), ("call","assigned",7,275), ("put","exercise",2,320), ("put","assigned",4,340)]:
    print(f"  {role} {n} x {X} {kind}: cash {cash_settle(kind, role, n, X, 300):>+9,}")

Cash (index 300, 500 per point):
  exercise 3 x 250 call: cash   +75,000
  assigned 7 x 275 call: cash   -87,500
  exercise 2 x 320 put: cash   +20,000
  assigned 4 x 340 put: cash   -80,000


## Try it
- Change the futures price in the futures section and see that the variation for an exerciser is the option's intrinsic value times the point value.
- Move the index to 250 and see which index options stop having value.